<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/stage_06_00_model_training_plan.ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_06 – Model Training – Objetivo y Modelos**


# **1. Plan de entrenamiento**

En este notebook se establecerá el plan de entrenamiento de modelos predictivos sobre el MNQ, manteniendo un esquema experimental consistente y comparable.

**Targets**

Se evaluarán cuatro variables objetivo:

- `delta_60`
- `delta_90`
- `ret_60`
- `ret_90`

**Window size**

Se utilizarán distintos tamaños de ventana histórica: `[30, 60 , 90, 120, 180]`
Cada ventana representa la cantidad de minutos utilizados como contexto de entrada.

**Features**

Todas las ventanas cuentan con 36 features de predicción:

```python
features_to_windows = [
    'minute_of_day',
    'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_closed',
    'is_mon', 'is_tue', 'is_wed', 'is_thu','is_fri',
    'close',
    'atr_norm_14', 'atr_norm_14_flag', 'atr_norm_20', 'atr_norm_20_flag',
    'ema_60', 'ema_60_flag',
    'mom_10', 'mom_10_flag', 'mom_5', 'mom_5_flag',
    'roc_20', 'roc_20_flag', 'roc_30', 'roc_30_flag', 'roc_60', 'roc_60_flag',
    'roc60_x_atr20', 'roc60_x_atr20_flag',
    'roc60_x_atr14', 'roc60_x_atr14_flag',
    'roc20_minus_roc60', 'roc20_minus_roc60_flag',
    'mom5_minus_mom10', 'mom5_minus_mom10_flag',
    ]
```

**Enfoques de predicción**

- **`seq2one`**: la secuencia de entrada genera una única predicción.
- **`seq2seq`**: la secuencia de entrada genera una secuencia de salida.

**Objetivo**

Comparar sistemáticamente:
- Puntos vs retornos  
- Sensibilidad al tamaño de ventana  
- Diferencias entre enfoques `many-to-one` y `many-to-many`

# **2. Estructura dimensional del problema de predicción**

## **2.1. Ventana de entrada $X$**

Para ambos enfoques de predicción, las ventanas de entrada $X$ tienen la siguiente dimensión:

$$
L \times N
$$

donde:

- $L$ representa la longitud de la ventana histórica y puede tomar los valores $[30, 60, 90, 120, 180]$.
- $N$ corresponde al número de variables predictoras (features), que en todos los casos es igual a **36**.


## **2.2 Ventana de salida $y$**

### **2.2.1. Enfoque `seq2one`**


En este esquema se entrenarán modelos **seq2one** para predecir un **único valor escalar** asociado a cada ventana histórica.

- **Salida / Target (Y)**

  $$
  Y \in \mathbb{R}^{1}
  $$

  Este valor representa la variable objetivo futura definida en los stages previos.

- **Interpretación**

  El modelo aprende una función del tipo:

  $$
  f: \mathbb{R}^{L \times N} \rightarrow \mathbb{R}
  $$

  Es decir, a partir de una ventana histórica de tamaño $L \times N$, el modelo produce **una única predicción final**.

  Este enfoque es adecuado cuando el objetivo es predecir el retorno o delta acumulado a un horizonte fijo $H$.

### **2.2.2. Enfoque `seq2seq`**


En este esquema se entrenarán modelos **seq2seq** para predecir una **secuencia futura completa** a partir de una ventana histórica intradía.

- **Salida / Target (Y)**

  $$
  Y \in \mathbb{R}^{L \times 1}
  $$

  donde:

  - $L$ es la longitud de la secuencia predicha.
  - $1$ corresponde a una variable objetivo escalar por cada paso temporal.

- **Interpretación**

  El modelo aprende una función del tipo:

  $$
  f: \mathbb{R}^{L \times N} \rightarrow \mathbb{R}^{L \times 1}
  $$

  En este caso, el modelo no predice un único valor final, sino la **trayectoria temporal completa** de $L$ pasos futuros.

  Este enfoque es adecuado cuando se desea modelar la dinámica temporal completa del horizonte futuro, y no únicamente su valor agregado final.



# **3. Alcance y criterios metodológicos del stage_06**

## **3.1. Qué se evalúa en este stage**

En el **stage_06** no se realiza la comparación final entre modelos ni se selecciona el “mejor modelo”.

Este stage garantiza que:

- Cada modelo es entrenado correctamente.
- Se respetan estrictamente los splits temporales definidos.
- La complejidad, arquitectura y regularización están fijadas antes del entrenamiento.
- El proceso es reproducible y comparable entre configuraciones.

La comparación real y la evaluación definitiva se realizan recién en el **stage_07**.

**Nota:** El set de validation puede utilizarse únicamente para control de entrenamiento (por ejemplo, early stopping o verificación de convergencia), pero no para seleccionar modelos.



## **3.2. Qué NO es este problema**


Quedan explícitamente fuera de alcance en este stage:

- Predicción de trayectorias completas multi-step adicionales fuera del esquema formal definido.
- Targets agregados ad-hoc (suma, promedio o retornos acumulados recalculados fuera del target previamente definido).
- Entrenamiento de múltiples regresiones independientes por paso futuro sin estructura temporal compartida.
- Modificaciones del esquema `seq2one` o `seq2seq` fuera de la formulación establecida.

Este stage se limita exclusivamente a entrenar modelos bajo una formulación bien definida y controlada del problema.



# **4.  Protocolo experimental común de entrenamiento**

Se entrenan múltiples modelos bajo reglas estrictamente comunes, garantizando comparabilidad entre configuraciones:

- Mismo dataset.
- Mismo split temporal (train / valid / test).
- Mismo escalamiento.
- Misma definición del target.
- Misma función objetivo.

Ejemplos típicos (alineados al enfoque del libro):

- Naive (baseline)
- Modelos lineales (Ridge, Lasso)
- Árboles y ensembles
- Redes neuronales (MLP, LSTM, etc.)

Cada modelo:

- Se entrena utilizando **exclusivamente el set de entrenamiento**.
- Tiene hiperparámetros y regularización definidos previamente.
- No se realizan ajustes posteriores basados en métricas del set de test.

El objetivo de este protocolo es asegurar que cualquier diferencia observada en desempeño se deba al modelo y no a variaciones en el proceso experimental.


# **5. Modelos a evaluar (comparación escalonada)**


Se define un baseline obligatorio para establecer un piso mínimo de desempeño y detectar rápidamente sobreajuste o complejidad innecesaria.

La evaluación se realiza de manera escalonada, comenzando por modelos simples y aumentando progresivamente la capacidad.

Se evaluarán varios modelos combinando:

- 4 targets (`delta_60`, `delta_90`, `ret_60`, `ret_90`)
- 5 window_size (`[30, 60, 90, 120, 180]`)
- Múltiples familias de modelos (baseline, lineales, recurrentes, convolucionales y atención)


## **5.1. Enfoque `seq2one`**

### **5.1.1. Modelos propuestos**

#### **(A) Baselines obligatorios**


1. **Naive / Persistence**

    - Predice el último valor observado de la ventana.
    - Define el piso mínimo de performance.
    - Modelo de control.
    - Notebook: `stage_07_01_naive_seq2one.ipynb`


#### **(B) Modelos clásicos**


2. **Ridge / Lasso**
    - Modelos lineales sobre la ventana aplanada (60x20 → 1200).
    - Referencia interpretable y estable.
    - Notebook: `stage_07_02_ridge_seq2one.ipynb`
    - Notebook: `stage_07_03_lasso_seq2one.ipynb`

3. **MLP (Feedforward)**
    - Entrada: ventana aplanada.
    - Salida: escalar.
    - Primera referencia no lineal.
    - Notebook: `stage_07_04_mlp_seq2one.ipynb`



#### **(C) Modelos temporales (many-to-one)**


4. **LSTM many-to-one**

    - Encoder recurrente.
    - Se utiliza únicamente el último estado oculto.
    - Salida escalar.
    - Notebook: `stage_07_05_lstm_seq2one.ipynb`

5. **GRU many-to-one**

    - Variante más simple y estable que LSTM.
    - Notebook: `stage_07_06_gru_seq2one.ipynb`

#### **(D) Modelo no recurrente robusto**


6. **TCN many-to-one**

    - Convoluciones causales dilatadas.
    - Último timestep → proyección escalar.
    - Muy buena estabilidad intradía.
    - Notebook: `stage_07_07_tcn_seq2one.ipynb`


#### **(E)  Modelo con atención**


7. **Transformer encoder-only (seq2one)**

    - Solo encoder.
    - Pooling o token final → salida escalar.
    - Mayor capacidad, mayor riesgo.
    - Notebook: `stage_07_08_transformer_seq2one.ipynb`

### **5.1.2. Regularización - `seq2one`**

La regularización forma parte del diseño de cada modelo y es clave para controlar
la capacidad y garantizar generalización.  
Cada familia de modelos requiere un esquema distinto.

#### **(A) Baselines obligatorios**


1. **Naive / Persistence**

    **Regularización:** No aplica.

    - No tiene parámetros entrenables.
    - No puede sobreajustar en sentido de Machine Learning.
    - Rol: define el piso mínimo de performance.
    - Si un modelo entrenable no supera este baseline en validation, se descarta.

#### **(B) Modelos clásicos**


2. **Ridge / Lasso**

    **Riesgo:** Bajo, controlado por diseño.

    **Regularización recomendada:**

    - Penalización L2 (Ridge) o L1 (Lasso) como mecanismo principal.
    - Selección del coeficiente de regularización previa al entrenamiento.
    - No requiere dropout ni early stopping.

    **Rol:** referencia lineal, estable e interpretable.

3. **MLP (Feedforward, seq2one)**

    **Riesgo principal:** Sobreajuste por capacidad del modelo.

    **Regularización recomendada:**

    - Penalización L2 (weight decay) como mecanismo principal.
    - Early stopping monitoreando la pérdida en validation.
    - Arquitectura limitada:
      - pocas capas (1–2),
      - número reducido de neuronas.
    - Dropout leve (opcional, solo si se observa sobreajuste claro).

    **Interpretación:** modelo no lineal flexible que requiere regularización explícita.


#### **(C) Modelos temporales (many-to-one)**


4. **LSTM many-to-one**

    **Riesgo:** Alta capacidad combinada con memoria de largo plazo.

    **Regularización recomendada:**

    - Early stopping obligatorio.
    - Tamaño moderado del hidden state.
    - Número de capas limitado (1–2).
    - Dropout en entradas y salidas (no recurrente).
    - Penalización L2 suave (opcional).

    **Nota:** se utiliza únicamente el último estado oculto para la predicción escalar.


5. **GRU many-to-one**

    **Riesgo:** Menor que LSTM, pero presente.

    **Regularización recomendada:**

    - Early stopping como mecanismo principal.
    - Limitar dimensión del hidden state.
    - Limitar número de capas.
    - Dropout opcional entre capas (no dentro de la recurrencia).

    **Ventaja:** mayor estabilidad y menor necesidad de regularización que LSTM.


#### **(D) Modelo no recurrente robusto**


6. **TCN (Temporal Convolutional Network, seq2one)**

    **Riesgo:** Campo receptivo excesivo o filtros redundantes.

    **Regularización recomendada:**

    - Limitar profundidad del modelo (dilataciones).
    - Limitar número de filtros por capa.
    - Dropout (especialmente efectivo en TCN).
    - Early stopping.
    - Weight normalization (si está disponible).

    **Ventaja clave:** buena generalización intradía con menor complejidad recurrente.


#### **(E) Modelo con atención**


7. **Transformer encoder-only (seq2one)**

    **Riesgo:** Sobreajuste por alta capacidad.

    **Regularización obligatoria:**

    - Dropout en bloques de atención y feed-forward.
    - Early stopping estricto.
    - Limitar número de capas del encoder.
    - Limitar dimensión del embedding.
    - Pooling simple o token final para salida escalar.

    **Regla práctica:** sin regularización fuerte, el modelo no generaliza.

#### **Resumen operativo**


- **Naive:** sin regularización.
- **Ridge / Lasso:** regularización integrada (L1 / L2).
- **MLP:** L2 + early stopping (+ dropout opcional).
- **GRU / LSTM:** early stopping + tamaño controlado + dropout.
- **TCN:** control de profundidad + dropout.
- **Transformer encoder:** dropout fuerte + límites estrictos.

La regularización no es un agregado opcional:  
define la capacidad efectiva del modelo y su comportamiento fuera de muestra.

La comparación final entre modelos se realiza recién en el **stage_07**.

## **5.2. Enfoque `seq2seq`**

### **5.2.1. Modelos propuestos**

#### **(A) Baselines obligatorios**


1. **Naive / Persistence**
   - Predice “sin cambio” (por ejemplo, replica el último valor observado) en los 29 pasos.
   - Define el “piso” mínimo de performance.
   - Notebook: `stage_07_01_naive_model.ipynb`

2. **MLP (Direct Multi-step)**
   - Aplana $(29 \times 8)$ y predice $(29 \times 1)$.
   - Base neural simple y rápida para validar el pipeline.
   - Notebook: `stage_07_02_mlp_direct_multistep.ipynb`

#### **(B) Seq2Seq clásicos**


3. **Encoder–Decoder GRU**
    - Notebook: `stage_07_03_gru_seq2seq.ipynb`

4. **Encoder–Decoder LSTM**
    - Encoder resume la historia; decoder genera la secuencia futura.
    - Estables y comparables para intradía.
    - Notebook: `stage_07_04_lstm_seq2seq.ipynb`



#### **(C) Alternativa robusta no recurrente**


5. **TCN (Temporal Convolutional Network)**
   - Convoluciones causales dilatadas.
   - Buena relación performance/estabilidad.
   - Notebook: `stage_07_05_tcn_seq2seq.ipynb`

#### **(D) Modelo de atención**


6. **Transformer Seq2Seq**
   - Encoder–decoder con atención.
   - Requiere regularización y control cuidadoso, pero es candidato fuerte.
   - Notebook: `stage_07_06_transformer_seq2seq.ipynb`

#### **(E) Modelo avanzado con estructura temporal explícita**


7. **Temporal Fusion Transformer (TFT)**

- Modelo seq2seq con atención para series temporales multivariadas.
- Incorpora selección de variables y atención temporal para capturar dependencias pasadas y futuras.
- Alta capacidad para modelar patrones intradía complejos; referencia avanzada frente a LSTM, TCN y Transformer estándar.
- Notebook: `stage_07_07_tft_seq2seq.ipynb`

### **5.2.2. Regularización de modelos**

#### **(A) Baselines obligatorios**


1. **Naive / Persistence**

    Regularización:
    No aplica.

    - No tiene parámetros entrenables.
    - No puede sobreajustar en sentido de Machine Learning.

    Rol:
    Define el piso mínimo de performance.
    Si un modelo entrenable no supera este baseline en validation, se descarta.

2. **MLP (Direct Multi-step)**

    Riesgo principal:
    Sobreajuste por capacidad del modelo.

    Regularización recomendada:

    - Penalización L2 (weight decay) como mecanismo principal.
    - Early stopping monitoreando la pérdida en validation.
    - Arquitectura limitada:
      - pocas capas (1–2),
      - número reducido de neuronas.
    - Dropout leve (opcional, solo si se observa sobreajuste claro).

    Interpretación según el libro:
    Modelo flexible que requiere regularización explícita para generalizar.

#### **(B) Seq2Seq clásicos**


3. **Encoder–Decoder GRU**

    Riesgo:
    Memorización de secuencias específicas del conjunto de entrenamiento.

    Regularización recomendada:

    - Early stopping como mecanismo principal.
    - Limitar la dimensión del hidden state.
    - Limitar el número de capas (1–2).
    - Dropout opcional entre capas, no dentro de la recurrencia.

    Nota:
    GRU es más estable que LSTM y suele requerir menor regularización.

4. **Encoder–Decoder LSTM**

    Riesgo:
    Alta capacidad combinada con memoria de largo plazo.

    Regularización recomendada:

    - Early stopping obligatorio.
    - Dropout en entradas y salidas (no recurrente).
    - Tamaño moderado del hidden state.
    - Número de capas limitado.
    - Penalización L2 suave en los pesos (opcional).

    Interpretación según el libro:
    Modelo potente que requiere regularización estructural y temporal.


#### **(C) Alternativa robusta no recurrente**


5. **TCN (Temporal Convolutional Network)**

    Riesgo:
    Campo receptivo excesivo o filtros redundantes.

    Regularización recomendada:

    - Limitar el número de filtros por capa.
    - Limitar la profundidad del modelo (dilataciones).
    - Dropout (especialmente efectivo en TCN).
    - Weight normalization si está disponible.
    - Early stopping.

    Ventaja clave:
    Suele generalizar mejor que modelos recurrentes con menor regularización agresiva.

#### **(D) Modelo de atención**


6. **Transformer Seq2Seq**

    Riesgo:
    Sobreajuste severo debido a alta capacidad.

    Regularización obligatoria:

    - Dropout alto en bloques de atención y feed-forward.
    - Early stopping estricto.
    - Limitar el número de capas.
    - Limitar la dimensión del embedding.
    - Label smoothing opcional en esquemas de pérdida multi-step.

    Regla práctica:
    Sin regularización fuerte, el modelo no generaliza.

#### **(E) Modelo avanzado con estructura temporal explícita**


7. **Temporal Fusion Transformer (TFT)**

    Riesgo:
    Alta capacidad, parcialmente mitigada por su diseño estructurado.

    Regularización integrada en la arquitectura:

    - Redes de selección de variables.
    - Mecanismos de gating.
    - Dropout configurable.
    - Early stopping.

    Aspectos que deben controlarse explícitamente:

    - Dimensión del hidden state.
    - Número de capas LSTM internas.
    - Nivel de dropout global.

    Interpretación según el libro:
    Modelo avanzado que regulariza principalmente por arquitectura, no solo por penalización.

#### **Resumen operativo**


- Naive: sin regularización.
- MLP: L2 + early stopping.
- GRU / LSTM: early stopping + tamaño controlado + dropout.
- TCN: control de profundidad + dropout.
- Transformer: dropout fuerte + límites estrictos.
- TFT: regularización arquitectónica + early stopping.

La regularización no es un agregado opcional.
Forma parte del diseño del modelo y determina su capacidad de generalización.

Cada modelo requiere un esquema de regularización distinto.
La comparación real entre ellos se realiza recién en el siguiente stage_07

## **5.3. Modelos según estructura de ventana**

### **5.3.1. Modelos con ventanas 2D (ventana aplanada)**

 Entrada:
  ```python
  (n_samples, window_size * n_features)
  ```
  No modela estructura temporal explícita.
  
  **Modelos:**
  - Naive (seq2one)
  - Ridge
  - Lasso
  - MLP (seq2one)
  - MLP Direct Multi-step (seq2seq)

### **5.3.2. Modelos con ventanas 3D (estructura temporal explícita)**

Entrada:
  
  ```python
  (n_samples, window_size, n_features)
  ```

Conserva la dimensión temporal y permite modelar dependencias dinámicas.

**Modelos:**
- Naive (seq2seq)
- LSTM (seq2one y seq2seq)
- GRU (seq2one y seq2seq)
- TCN (seq2one y seq2seq)
- Transformer encoder-only (seq2one)
- Transformer Seq2Seq
- Temporal Fusion Transformer (TFT)

# **6. Definición de la comparación Predicción vs Valor Real**

En este stage, la evaluación del modelo se realiza bajo los esquemas **`seq2one`** y **`seq2seq`**.  
Cada ventana histórica produce:

- una predicción escalar (seq2one), o  
- una secuencia futura completa (seq2seq).

La comparación siempre se realiza directamente entre predicción y valor real correspondiente, respetando la formulación original del problema.



## **6.1 Esquema de predicción**

### **6.1.1. Enfoque `seq2one``**

Para cada muestra $t$, el modelo recibe como entrada:

$$
X_t \in \mathbb{R}^{L \times N}
$$

correspondiente a $L$ minutos consecutivos con $N$ features por minuto.

El modelo produce una única predicción escalar:

$$
\hat{y}_t \in \mathbb{R}
$$

que representa el valor futuro del target definido (por ejemplo, $\Delta_h$ para un horizonte fijo $h$).

Se compara directamente contra el valor real observado:

$$
y_t \in \mathbb{R}
$$

La comparación es **escalar contra escalar**, sin:

- generar trayectorias,
- aplicar agregaciones intermedias,
- ni comparar secuencias completas.

Cada ventana histórica tiene una única predicción y un único valor real asociado.


### **6.1.2. Enfoque `seq2seq`**

Para cada muestra $t$, el modelo recibe como entrada:

$$
X_t \in \mathbb{R}^{L \times N}
$$

A partir de esta entrada, el modelo predice una secuencia futura completa:

$$
\hat{Y}_{t+1:t+L} \in \mathbb{R}^{L \times 1}
$$

es decir, un valor del target por cada uno de los $L$ pasos futuros.

La predicción se compara directamente contra la secuencia real futura observada:

$$
Y^{real}_{t+1:t+L} \in \mathbb{R}^{L \times 1}
$$

La comparación es **secuencia contra secuencia**, sin colapsar el target ni aplicar agregaciones previas.


## **6.2 Cálculo de métricas**

### **6.2.1. Enfoque `seq2one``**

A partir de la comparación $\hat{y}_t$ vs $y_t$, las métricas se calculan
sobre el conjunto completo de muestras del split correspondiente.

Métricas utilizadas:

- MAE
- RMSE
- R² (opcional)
- Métricas direccionales (signo de la predicción vs signo real)

Las métricas se agregan sobre todas las ventanas del split.

### **6.2.2. Enfoque `seq2seq`**


Las métricas se calculan bajo dos perspectivas:

**(A) Por paso temporal**

Comparación entre:

$$
\hat{y}_{t+k} \quad \text{vs} \quad y^{real}_{t+k}, \quad k = 1, \dots, L
$$

**(B) Sobre la trayectoria completa**

Error global entre:

$$
\hat{Y}_{t+1:t+L} \quad \text{vs} \quad Y^{real}_{t+1:t+L}
$$

No se realiza comparación contra un escalar ni contra un valor agregado final.

El problema se mantiene estrictamente bajo la formulación `seq2seq`.

## **6.3. Uso de los splits (criterio de evaluación)**


Siguiendo el criterio operativo del workflow:

- **TRAIN**: utilizado exclusivamente para el aprendizaje del modelo.
- **VALID**: utilizado para medir desempeño y descartar modelos que no generalizan.
- **TEST**: utilizado únicamente una vez finalizada la selección del modelo.

No se ajustan hiperparámetros en función del set de test.

El modelo se entrena solo con TRAIN, pero se evalúa con VALID para decidir si es candidato a pasar al stage siguiente.

En términos operativos:

- TRAIN → para aprender.
- VALID → para medir desempeño y comparar modelos.
- TEST → solo al final, una vez elegido el mejor modelo.

Este esquema evita contaminación de información y garantiza evaluación fuera de muestra.


## **6.4. Derivación de métricas**


A partir de la comparación entre predicción y valor real (según el esquema `seq2one` o `seq2seq`), se derivan dos grupos de métricas:

- **(A) Métricas de Machine Learning**

  - MAE
  - RMSE
  - R² (opcional)
  - Métricas direccionales (signo de la predicción vs signo real)

  Estas métricas se calculan sobre VALID durante el stage_06 para decidir qué modelos continúan.

- **(B) Métricas económicas**

  Una vez seleccionados los modelos candidatos, se derivan métricas económicas utilizando el delta real observado dentro de la ventana futura:

  - Valor esperado (EV)
  - Ratio TP/SL
  - Drawdown
  - Métricas de riesgo-retorno

  Estas métricas permiten traducir el desempeño estadístico en impacto operativo.

La comparación económica definitiva se realiza posteriormente en el stage_07.

# **7. Métricas de predicción (Machine Learning)**

La evaluación del desempeño se realiza exclusivamente fuera de muestra (VALID).  
El conjunto TEST se reserva únicamente para la evaluación final una vez seleccionado el modelo.

Las métricas se organizan según:

- Métricas principales (criterio de selección)
- Métricas complementarias (interpretación)
- Métricas diagnósticas (solo análisis interno)

## **7.1. Enfoque `seq2one`**

Dado que el problema consiste en la predicción de un valor escalar futuro, la comparación se realiza entre:

$$
\hat{y}_t \quad \text{vs} \quad y_t
$$

### **7.1.1 Métricas principales (criterio de selección)**

**1. MAE (Mean Absolute Error)**

Error absoluto medio entre predicción y valor real:

$$
\text{MAE} = \frac{1}{N} \sum_{t=1}^{N} |\hat{y}_t - y_t|
$$

**2. RMSE (Root Mean Squared Error)**

Raíz del error cuadrático medio:

$$
\text{RMSE} = \sqrt{\frac{1}{N} \sum_{t=1}^{N} (\hat{y}_t - y_t)^2}
$$

Estas métricas constituyen el **criterio principal de comparación y ranking** de modelos.



### **7.1.2 Métricas complementarias**


**3. Directional Accuracy (DA)**

$$
DA = P\left[\text{sign}(\hat{y}_t) = \text{sign}(y_t)\right]
$$

- No se utiliza como criterio principal de selección.
- Se reporta con fines interpretativos.
- Conecta la predicción con la dirección esperada del movimiento.

**4. Coeficiente de determinación ($R^2$)**

- Se reporta como métrica descriptiva.
- No se utiliza como criterio principal debido a su limitada estabilidad en series financieras.


## **7.2. Enfoque `seq2seq`**

Dado que el problema consiste en la predicción de una secuencia futura completa, la comparación se realiza entre:

$$
\hat{Y}_{t+1:t+L} \quad \text{vs} \quad Y_{t+1:t+L}
$$

### **7.2.1 Métricas principales (criterio de selección)**

Las métricas se calculan de forma global concatenando todos los pasos de la secuencia en un único vector.

**1. MAE global**

- Error absoluto medio sobre todos los pasos y todas las muestras.

**2. RMSE global**

- Raíz del error cuadrático medio sobre todos los pasos y todas las muestras.

Estas métricas constituyen el criterio principal de comparación y ranking de modelos.


### **7.2.2 Métricas diagnósticas**


**3. Error por horizonte**

$$
MAE(k), \quad k = 1, \dots, L
$$

Permite:

- Observar la degradación del error a medida que aumenta el horizonte.
- Analizar la estabilidad temporal del modelo.

Este análisis es estrictamente diagnóstico y no se utiliza para selección final.



### **7.2.3 Métricas complementarias**


**4. Directional Accuracy (DA_last)**

Coincidencia de signo en el último paso de la secuencia:

$$
k = L
$$

- Conecta con la dirección esperada al horizonte final.
- Se utiliza solo con fines interpretativos.

**5. Coeficiente de determinación ($R^2$)**

- Se reporta como métrica descriptiva.
- No se utiliza como criterio principal de comparación.



## **7.3 Jerarquía de métricas**

**Métricas principales (criterio de selección)**
- MAE
- RMSE

**Métricas complementarias**
- Directional Accuracy (DA o DA_last)
- R²

**Métricas diagnósticas (solo seq2seq)**
- MAE(k)

La selección y descarte de modelos se realiza exclusivamente en base a las métricas principales evaluadas sobre VALID.


## **7.4 Función de cálculo de métricas**

### **7.4.1. Función de cálculo de métricas seq2one**

In [ ]:
import numpy as np
from sklearn.metrics import r2_score

def compute_seq2one_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    *,
    compute_r2: bool = True,
    da_ignore_zeros: bool = True,
    allow_seq_inputs_take_last: bool = False,
) -> dict:
    """
    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    """

    # 1) Convertir a np.ndarray y forzar float
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    # 2) Normalizar dimensiones hacia (n_samples,)
    def _to_1d(y: np.ndarray, name: str) -> np.ndarray:
        if y.ndim == 1:
            return y
        if y.ndim == 2:
            # (n, 1) -> (n,)
            if y.shape[1] == 1:
                return y.squeeze(1)
            # (n, seq_len) -> tomar último si se permite
            if allow_seq_inputs_take_last:
                return y[:, -1]
            raise ValueError(
                f"{name} con shape {y.shape} no es válido para seq2one. "
                f"Se esperaba (n_samples,) o (n_samples, 1)."
            )
        if y.ndim == 3 and y.shape[-1] == 1:
            # (n, seq_len, 1) -> (n, seq_len) y luego último si se permite
            y2 = y.squeeze(-1)
            if allow_seq_inputs_take_last:
                return y2[:, -1]
            raise ValueError(
                f"{name} con shape {y.shape} parece seq2seq. "
                f"Active allow_seq_inputs_take_last=True si quiere tomar el último paso."
            )
        raise ValueError(
            f"{name}.ndim={y.ndim} no es válido. "
            f"Se esperaba (n,), (n,1) o (n,seq_len) si allow_seq_inputs_take_last=True."
        )

    y_true = _to_1d(y_true, "y_true")
    y_pred = _to_1d(y_pred, "y_pred")

    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"y_true y y_pred deben tener el mismo shape. "
            f"Recibido y_true={y_true.shape}, y_pred={y_pred.shape}"
        )

    n_samples = int(y_true.shape[0])
    if n_samples == 0:
        raise ValueError("y_true/y_pred no pueden estar vacíos.")

    # 3) Validación numérica básica
    if not (np.isfinite(y_true).all() and np.isfinite(y_pred).all()):
        raise ValueError("Se encontraron NaN o inf en y_true/y_pred. "
                         "Limpie o enmascare antes de calcular métricas.")

    # 4) Errores
    errors = y_pred - y_true
    abs_errors = np.abs(errors)

    # 5) Métricas principales
    mae = float(abs_errors.mean())
    rmse = float(np.sqrt((errors ** 2).mean()))

    # 6) Métrica direccional (DA)
    sign_true = np.sign(y_true)
    sign_pred = np.sign(y_pred)

    if da_ignore_zeros:
        mask = (sign_true != 0) & (sign_pred != 0)
        da = float(np.mean(sign_true[mask] == sign_pred[mask])) if mask.any() else float("nan")
        da_n = int(mask.sum())
    else:
        da = float(np.mean(sign_true == sign_pred))
        da_n = n_samples

    metrics = {
        "MAE": mae,
        "RMSE": rmse,
        "DA": da,
        "DA_n": da_n,  # cuántas muestras realmente aportaron a DA (si ignore_zeros=True)
    }

    # 7) R² opcional
    if compute_r2:
        metrics["R2"] = float(r2_score(y_true, y_pred))

    return metrics

### **7.4.2. Función de cálculo de métricas seq2seq**

In [ ]:
import numpy as np
from sklearn.metrics import r2_score

def compute_seq2seq_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    compute_r2: bool = True,
    da_ignore_zeros: bool = True,
) -> dict:
    """
    Calcula métricas comparables para modelos seq2seq.

    Retorna:
    - Métricas globales (MAE, RMSE)
    - Métricas del último paso (MAE_last, RMSE_last)
    - DA_last
    - MAE por paso (diagnóstico)
    """

    # 1) Convertir a np.ndarray
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    # 2) Normalizar dimensiones a (n_samples, seq_len)
    if y_true.ndim == 3 and y_true.shape[-1] == 1:
        y_true = y_true.squeeze(-1)
    if y_pred.ndim == 3 and y_pred.shape[-1] == 1:
        y_pred = y_pred.squeeze(-1)

    if y_true.ndim != 2 or y_pred.ndim != 2:
        raise ValueError(
            f"Se espera shape (n_samples, seq_len) o (n_samples, seq_len, 1). "
            f"Recibido y_true.ndim={y_true.ndim}, y_pred.ndim={y_pred.ndim}"
        )

    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"y_true y y_pred deben tener el mismo shape. "
            f"Recibido y_true={y_true.shape}, y_pred={y_pred.shape}"
        )

    n_samples, seq_len = y_true.shape
    if n_samples == 0 or seq_len == 0:
        raise ValueError("y_true/y_pred no pueden estar vacíos.")

    # 3) Validación numérica
    if not (np.isfinite(y_true).all() and np.isfinite(y_pred).all()):
        raise ValueError("Se encontraron NaN o inf en y_true/y_pred.")

    # 4) Errores
    errors = y_pred - y_true
    abs_errors = np.abs(errors)

    # ===============================
    # MÉTRICAS GLOBALES (seq2seq puro)
    # ===============================
    mae = float(abs_errors.mean())
    rmse = float(np.sqrt((errors ** 2).mean()))

    # ===============================
    # MÉTRICAS ÚLTIMO PASO (comparables con seq2one)
    # ===============================
    y_true_last = y_true[:, -1]
    y_pred_last = y_pred[:, -1]

    errors_last = y_pred_last - y_true_last

    mae_last = float(np.abs(errors_last).mean())
    rmse_last = float(np.sqrt((errors_last ** 2).mean()))

    # ===============================
    # Directional Accuracy (último paso)
    # ===============================
    sign_true = np.sign(y_true_last)
    sign_pred = np.sign(y_pred_last)

    if da_ignore_zeros:
        mask = (sign_true != 0) & (sign_pred != 0)
        da_last = float(np.mean(sign_true[mask] == sign_pred[mask])) if mask.any() else float("nan")
        da_last_n = int(mask.sum())
    else:
        da_last = float(np.mean(sign_true == sign_pred))
        da_last_n = int(n_samples)

    # ===============================
    # MAE por paso (diagnóstico)
    # ===============================
    mae_per_step = abs_errors.mean(axis=0)  # (seq_len,)

    metrics = {
        # Global seq2seq
        "MAE": mae,
        "RMSE": rmse,

        # Último paso (comparables con seq2one)
        "MAE_last": mae_last,
        "RMSE_last": rmse_last,

        # Dirección
        "DA_last": da_last,
        "DA_last_n": da_last_n,

        # Diagnóstico
        "MAE_per_step": mae_per_step.tolist(),
    }

    # ===============================
    # R² opcional
    # ===============================
    if compute_r2:
        metrics["R2"] = float(r2_score(y_true.ravel(), y_pred.ravel()))
        metrics["R2_last"] = float(r2_score(y_true_last, y_pred_last))

    return metrics

In [ ]:
#Ejemplo de uso:
#metrics = compute_seq2seq_metrics(y_true, y_pred)
#print(metrics["MAE"], metrics["RMSE"], metrics["DA_last"])

# **8. Métricas de SEQ2ONE**

In [9]:
import pandas as pd
from pathlib import Path
import os

In [10]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import pandas as pd
from pathlib import Path

def load_all_seq2one_metrics(
    *,
    models: list[str],
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
) -> pd.DataFrame:
    """
    Carga métricas seq2one para múltiples modelos y
    mantiene solo columnas estándar comparables.
    """

    cols = [
        "model", "split", "window_size", "target",
        "horizon_min", "MAE", "RMSE", "R2", "DA"
    ]

    dfs = []

    base_path = Path(base_dir)

    for name in models:
        path = base_path / f"seq2one_{name}_metrics.parquet"

        if not path.exists():
            continue

        df = pd.read_parquet(path)

        # Mantener solo columnas deseadas si existen
        keep_cols = [c for c in cols if c in df.columns]
        df = df[keep_cols].copy()

        dfs.append(df)

    if not dfs:
        return pd.DataFrame(columns=cols)

    df_all = pd.concat(dfs, ignore_index=True)

    # Orden consistente
    df_all = (
        df_all
        .sort_values(["model", "window_size", "target", "split"])
        .reset_index(drop=True)
    )

    return df_all


In [12]:
models_seq2one = ['naive', 'ridge', 'lasso', 'mlp', 'gru', 'lstm', 'tcn', 'transformer']

df_seq2one_all = load_all_seq2one_metrics(
    models=models_seq2one,
    base_dir="/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics"
)

In [5]:
df_seq2one_all

,model,split,window_size,target,horizon_min,MAE,RMSE,R2,DA
0,gru,test,30,delta_60,60,48.576339,76.975064,0.115642,0.619139
1,gru,valid,30,delta_60,60,30.210935,43.738906,0.208353,0.647863
2,gru,test,30,delta_90,90,62.513185,98.251515,0.078711,0.598776
3,gru,valid,30,delta_90,90,39.720030,57.446933,0.128378,0.624933
4,gru,test,30,ret_60,60,0.005739,0.008296,-2.881864,0.499769
...,...,...,...,...,...,...,...,...,...
251,transformer,valid,60,delta_90,90,34.308975,51.259851,0.309020,0.687362
252,transformer,test,60,ret_60,60,0.004160,0.011002,-5.756404,0.541288
253,transformer,valid,60,ret_60,60,0.002450,0.003312,-0.472759,0.554930
254,transformer,test,60,ret_90,90,0.003227,0.006247,-0.439008,0.624305


# Orden propuesto para selección de modelos SEQ2ONE

## 1. Selección del tipo de target

### Criterios primarios

- R² > 0 de forma consistente entre modelos.
- MAE estable y razonable.
- Directional Accuracy (DA) claramente superior a 0.55.
- Baja varianza de desempeño entre modelos.

### Observación preliminar

- `delta_60` y `delta_90` presentan R² positivos y razonables.
- `ret_60` y `ret_90` muestran R² negativos en la mayoría de los casos.

### Decisión metodológica

Primero se debe seleccionar entre `delta` y `ret`.

Si el objetivo es obtener una señal predictiva explotable, el target elegido debe:

- Presentar R² positivo en la mayoría de los modelos.
- Tener mayor DA promedio.
- Mostrar menor dispersión de resultados entre arquitecturas.

Preliminarmente, el candidato más sólido parece ser `delta_60`.

---

## 2. Selección del horizonte

Comparar:

- `delta_60`
- `delta_90`

### Criterios

- Mayor R² promedio entre modelos.
- Mejor DA promedio.
- Menor dispersión entre modelos.

Si `delta_60` domina en estabilidad y consistencia, se elige 60 minutos.  
Si `delta_90` muestra mejor robustez estructural, se elige 90 minutos.

---

## 3. Selección de window_size

Una vez fijado el target y horizonte:

Para cada `window_size` evaluar:

- Promedio de R² entre modelos.
- Mejor R² alcanzado.
- Consistencia (por ejemplo, cantidad de modelos con R² > 0.25).

### Criterio recomendado

- No elegir la ventana únicamente por el mejor modelo individual.
- Elegir la ventana con mejor desempeño agregado y estabilidad.

Esto reduce el riesgo de sobreajuste estructural.

---

## 4. Selección de los dos mejores modelos

Una vez definidos:

- target
- horizon
- window_size

Ordenar los modelos por:

1. R² (criterio principal)
2. MAE (criterio secundario)
3. DA (validación direccional)

Seleccionar:

- El mejor modelo absoluto.
- El segundo mejor modelo que sea estructuralmente diferente.

Ejemplo:  
- Transformer  
- GRU o MLP  

No es recomendable elegir dos modelos demasiado similares.

---

## Resumen del orden final

1. Elegir tipo de target (`delta` vs `ret`).
2. Elegir horizonte (60 vs 90).
3. Elegir `window_size`.
4. Elegir los dos mejores modelos.
5. Realizar tuning exclusivamente sobre VALID.

## **8.1. Selección de target**

# Criterio de decisión: uso de R² como métrica principal

## 1. Qué mide R²

R² mide la **proporción de varianza explicada** por el modelo respecto a un baseline constante.

En regresión:

R² = 1 − (MSE_model / MSE_baseline)

Donde el baseline es predecir siempre la media del target.

Interpretación:

- R² > 0 → el modelo mejora al baseline.
- R² = 0 → el modelo es equivalente al baseline.
- R² < 0 → el modelo es peor que el baseline.

Por lo tanto, R² es una métrica estructural que indica si existe capacidad explicativa real.

---

## 2. Relevancia para este problema

En esta etapa el objetivo no es aún maximizar PnL, sino:

> Evaluar si el target es predecible.

R² responde directamente a esa pregunta.

MAE y RMSE solo miden magnitud del error absoluto, pero no indican si el modelo mejora significativamente respecto a un baseline simple.

Un modelo puede tener MAE bajo y aun así no explicar varianza relevante si el target tiene poca dispersión.

---

## 3. Interpretación de R² en series financieras

En problemas financieros:

- R² ≈ 5% ya es interesante.
- R² ≈ 20–30% es muy fuerte.
- R² negativo implica ausencia de señal explotable.

Usar R² como criterio principal equivale a preguntar:

> ¿Existe señal estructural o estamos modelando ruido?

---

## 4. Por qué MAE no es la métrica principal

Limitaciones del MAE:

- Depende de la escala del target.
- No es relativo a un baseline.
- No permite comparar fácilmente targets con distinta varianza.

Ejemplo:

- delta_60 puede tener MAE = 25
- ret_60 puede tener MAE = 0.002

No son directamente comparables.

R² sí permite comparación transversal entre targets.

---

## 5. Por qué DA no puede ser el criterio central

Directional Accuracy (DA):

- Ignora magnitud del error.
- Puede inflarse si el target tiene sesgo estructural.
- No penaliza errores grandes.

DA es útil como métrica complementaria, pero no como criterio principal de selección.

---

## 6. Orden metodológico en esta etapa

Para la selección del tipo de target:

1. R² → evaluar capacidad explicativa.
2. DA → validar señal direccional.
3. MAE → analizar estabilidad del error.
4. Varianza del desempeño → evaluar robustez entre modelos.

---

## Conclusión

R² se utiliza como métrica principal porque:

- Es relativa al baseline.
- Mide capacidad explicativa real.
- Permite comparar targets con distinta escala.
- Responde a la pregunta fundamental: si existe señal estructural en el target.

In [16]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [13]:
import pandas as pd
import numpy as np

# -------------------------------------------------
# Configuración
# -------------------------------------------------
DF = df_seq2one_all.copy()
SPLIT = "valid"

# Filtrar solo VALID
df_valid = DF[DF["split"] == SPLIT].copy()

In [17]:
import pandas as pd
import numpy as np

# Requiere df_valid con columnas:
# ['model','split','window_size','target','horizon_min','MAE','RMSE','R2','DA']

required = {"model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA"}
missing = required - set(df_valid.columns)
if missing:
    raise ValueError(f"df_valid no tiene columnas requeridas: {sorted(missing)}")

df = df_valid.copy()
df = df[df["split"].astype(str).str.lower().eq("valid")].copy()

# Normalización mínima
df["target"] = df["target"].astype(str).str.lower().str.strip()

# Tipo de target: delta vs ret
df["target_type"] = np.where(df["target"].str.startswith("delta"), "delta",
                      np.where(df["target"].str.startswith("ret"), "ret", "other"))

# Flags de criterios primarios
df["r2_pos"] = df["R2"] > 0
df["da_gt_055"] = df["DA"] > 0.55

def summarize(g: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "n_rows": len(g),
        "n_models": g["model"].nunique(),
        "n_windows": g["window_size"].nunique(),

        "R2_mean": g["R2"].mean(),
        "R2_median": g["R2"].median(),
        "R2_std": g["R2"].std(ddof=0),
        "R2_share_pos": g["r2_pos"].mean(),

        "DA_mean": g["DA"].mean(),
        "DA_median": g["DA"].median(),
        "DA_std": g["DA"].std(ddof=0),
        "DA_share_gt_055": g["da_gt_055"].mean(),

        "MAE_mean": g["MAE"].mean(),
        "MAE_median": g["MAE"].median(),
        "MAE_std": g["MAE"].std(ddof=0),
    })

def fmt_table(t: pd.DataFrame) -> pd.DataFrame:
    out = t.copy()
    # porcentajes
    for c in ["R2_share_pos", "DA_share_gt_055"]:
        if c in out.columns:
            out[c] = (out[c] * 100).round(1).astype(str) + "%"
    # redondeo numérico
    for c in out.columns:
        if c not in ["target","target_type","model","n_rows","n_models","n_windows","R2_share_pos","DA_share_gt_055"]:
            if pd.api.types.is_numeric_dtype(out[c]):
                out[c] = out[c].round(4)
    return out

# -------------------------------------------------------------------
# 1) Tabla por target específico (delta_60, delta_90, ret_60, ret_90)
# -------------------------------------------------------------------
by_target = (
    df.groupby("target", as_index=False)
      .apply(lambda g: summarize(g))
      .reset_index(drop=True)
)

# Score para ordenar: prioriza R2 y DA, penaliza dispersión (R2_std) y MAE alta
by_target["score"] = (
    by_target["R2_mean"].rank(ascending=False, method="min") * 1.0 +
    by_target["DA_mean"].rank(ascending=False, method="min") * 0.7 +
    by_target["R2_std"].rank(ascending=True, method="min") * 0.6 +
    by_target["MAE_mean"].rank(ascending=True, method="min") * 0.3
)
by_target = by_target.sort_values(["score", "target"]).drop(columns=["score"])
by_target = fmt_table(by_target)

print("\n=== 1) Resumen por target (VALID) ===")
display(by_target)

# ------------------------------------------------------
# 2) Tabla por tipo de target: delta vs ret
# ------------------------------------------------------
by_type = (
    df[df["target_type"].isin(["delta","ret"])]
      .groupby("target_type", as_index=False)
      .apply(lambda g: summarize(g))
      .reset_index(drop=True)
      .sort_values("target_type")
)
by_type = fmt_table(by_type)

print("\n=== 2) Resumen por tipo de target: delta vs ret (VALID) ===")
display(by_type)

# -------------------------------------------------------------------
# 3) Comparación por modelo: delta vs ret (promedios por modelo)
# -------------------------------------------------------------------
model_type = (
    df[df["target_type"].isin(["delta","ret"])]
      .groupby(["model","target_type"], as_index=False)
      .apply(lambda g: pd.Series({
          "n_rows": len(g),
          "R2_mean": g["R2"].mean(),
          "DA_mean": g["DA"].mean(),
          "MAE_mean": g["MAE"].mean(),
          "R2_share_pos": (g["R2"] > 0).mean(),
          "DA_share_gt_055": (g["DA"] > 0.55).mean(),
          "R2_std": g["R2"].std(ddof=0),
      }))
      .reset_index(drop=True)
)

# Pivot para diferencias delta - ret (por modelo)
pivot_r2 = model_type.pivot(index="model", columns="target_type", values="R2_mean")
pivot_da = model_type.pivot(index="model", columns="target_type", values="DA_mean")
pivot_mae = model_type.pivot(index="model", columns="target_type", values="MAE_mean")

model_cmp = pd.DataFrame({
    "model": pivot_r2.index,
    "R2_mean_delta": pivot_r2.get("delta"),
    "R2_mean_ret": pivot_r2.get("ret"),
    "R2_delta_minus_ret": pivot_r2.get("delta") - pivot_r2.get("ret"),
    "DA_mean_delta": pivot_da.get("delta"),
    "DA_mean_ret": pivot_da.get("ret"),
    "DA_delta_minus_ret": pivot_da.get("delta") - pivot_da.get("ret"),
    "MAE_mean_delta": pivot_mae.get("delta"),
    "MAE_mean_ret": pivot_mae.get("ret"),
}).reset_index(drop=True)

model_cmp = model_cmp.sort_values("R2_delta_minus_ret", ascending=False)
for c in model_cmp.columns:
    if c != "model" and pd.api.types.is_numeric_dtype(model_cmp[c]):
        model_cmp[c] = model_cmp[c].round(4)

print("\n=== 3) Comparación por modelo: delta vs ret (VALID) ===")
display(model_cmp)

# ------------------------------------------------------
# Decisión sugerida (automática) delta vs ret
# ------------------------------------------------------
# Regla simple: elegir el tipo con:
# - mayor R2_mean
# - mayor DA_mean
# - mayor share R2>0
# - menor R2_std
delta_row = by_type[by_type["target_type"] == "delta"]
ret_row   = by_type[by_type["target_type"] == "ret"]

print("\n=== Decisión sugerida (criterios primarios) ===")
display(by_type)
print("Interpretación recomendada: elija el tipo con mayor R2_mean y DA_mean, mayor R2_share_pos y menor R2_std.")


=== 1) Resumen por target (VALID) ===


,target,n_rows,n_models,n_windows,R2_mean,R2_median,R2_std,R2_share_pos,DA_mean,DA_median,DA_std,DA_share_gt_055,MAE_mean,MAE_median,MAE_std
0,delta_60,32.0,7.0,5.0,0.2630,0.2619,0.0926,100.0%,0.6827,0.6938,0.0439,100.0%,28.0091,28.2422,2.7130
1,delta_90,32.0,7.0,5.0,0.2258,0.2183,0.0912,100.0%,0.6698,0.6715,0.0380,100.0%,36.4464,36.6836,2.9270
3,ret_90,32.0,7.0,5.0,-0.0808,-0.0014,0.3464,37.5%,0.6007,0.5787,0.0575,87.5%,0.0025,0.0024,0.0004
2,ret_60,32.0,7.0,5.0,-0.3157,-0.0006,1.1837,40.6%,0.5987,0.5825,0.0614,65.6%,0.0021,0.0019,0.0009



=== 2) Resumen por tipo de target: delta vs ret (VALID) ===


,target_type,n_rows,n_models,n_windows,R2_mean,R2_median,R2_std,R2_share_pos,DA_mean,DA_median,DA_std,DA_share_gt_055,MAE_mean,MAE_median,MAE_std
0,delta,64.0,7.0,5.0,0.2444,0.2354,0.0938,100.0%,0.6762,0.6866,0.0416,100.0%,32.2278,31.1274,5.0755
1,ret,64.0,7.0,5.0,-0.1982,-0.0008,0.8800,39.1%,0.5997,0.5787,0.0595,76.6%,0.0023,0.0022,0.0007



=== 3) Comparación por modelo: delta vs ret (VALID) ===


,model,R2_mean_delta,R2_mean_ret,R2_delta_minus_ret,DA_mean_delta,DA_mean_ret,DA_delta_minus_ret,MAE_mean_delta,MAE_mean_ret
3,mlp,0.3432,-0.9357,1.2789,0.7052,0.6307,0.0745,29.0198,0.0028
6,transformer,0.3110,-0.2105,0.5215,0.6798,0.5618,0.1180,31.0396,0.0025
5,tcn,0.1516,-0.3171,0.4687,0.6214,0.5593,0.0622,35.1010,0.0025
0,gru,0.2460,-0.1804,0.4264,0.6776,0.5974,0.0802,32.0576,0.0024
2,lstm,0.2050,-0.0509,0.2559,0.6598,0.5793,0.0806,33.0951,0.0022
1,lasso,0.1941,-0.0008,0.1949,0.6903,0.5499,0.1404,33.4777,0.0021
4,ridge,0.2999,0.3005,-0.0006,0.7015,0.6965,0.0050,31.0908,0.0017



=== Decisión sugerida (criterios primarios) ===


,target_type,n_rows,n_models,n_windows,R2_mean,R2_median,R2_std,R2_share_pos,DA_mean,DA_median,DA_std,DA_share_gt_055,MAE_mean,MAE_median,MAE_std
0,delta,64.0,7.0,5.0,0.2444,0.2354,0.0938,100.0%,0.6762,0.6866,0.0416,100.0%,32.2278,31.1274,5.0755
1,ret,64.0,7.0,5.0,-0.1982,-0.0008,0.8800,39.1%,0.5997,0.5787,0.0595,76.6%,0.0023,0.0022,0.0007


Interpretación recomendada: elija el tipo con mayor R2_mean y DA_mean, mayor R2_share_pos y menor R2_std.


# Selección del tipo de target (VALID)

## 1. Comparación delta vs ret

### Resultados agregados

| Métrica | delta | ret |
|----------|--------|--------|
| R2_mean | 0.2444 | -0.1982 |
| R2_std | 0.0938 | 0.8800 |
| R2_share_pos | 100% | 39.1% |
| DA_mean | 0.6762 | 0.5997 |
| DA_share_gt_055 | 100% | 76.6% |

---

## 2. Análisis estructural

### Capacidad explicativa (R²)

- Todos los modelos logran R² positivo en delta (100%).
- En ret, solo 39.1% de los casos tienen R² positivo.
- El R² promedio de ret es negativo (-0.1982).
- La dispersión en ret es extremadamente alta (R2_std = 0.88), indicando inestabilidad estructural.

Conclusión: delta presenta señal consistente; ret presenta comportamiento cercano a ruido.

---

### Señal direccional (DA)

- delta: DA_mean = 0.6762 (100% de los modelos > 0.55).
- ret: DA_mean = 0.5997 (76.6% > 0.55).

delta no solo explica varianza, sino que también mejora significativamente la predicción direccional.

---

### Robustez entre modelos

En la comparación por modelo:

- 6 de 7 modelos mejoran claramente al pasar de ret a delta.
- Solo ridge muestra comportamiento similar en ambos targets.
- El resto presenta mejoras sustanciales en R² y DA con delta.

Esto confirma que la superioridad de delta no depende de una arquitectura específica.

---

## 3. Conclusión metodológica

Bajo los criterios definidos:

- R² positivo de forma consistente.
- Mayor R² promedio.
- Menor dispersión (R2_std).
- Mayor DA promedio.
- Mayor proporción de modelos con desempeño sólido.

El tipo de target seleccionado es:

> **delta**

El target ret queda descartado en esta etapa por ausencia de capacidad explicativa robusta.

---

## Próximo paso

Dentro de delta, se procederá a comparar:

- delta_60
- delta_90

para seleccionar el horizonte óptimo.

In [18]:
import pandas as pd
import numpy as np

# Requiere df_valid con columnas:
# ['model','split','window_size','target','horizon_min','MAE','RMSE','R2','DA']

required = {"model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA"}
missing = required - set(df_valid.columns)
if missing:
    raise ValueError(f"df_valid no tiene columnas requeridas: {sorted(missing)}")

df = df_valid.copy()
df = df[df["split"].astype(str).str.lower().eq("valid")].copy()
df["target"] = df["target"].astype(str).str.lower().str.strip()

# Nos quedamos solo con delta_60 y delta_90
df = df[df["target"].isin(["delta_60", "delta_90"])].copy()

# Flags
df["r2_pos"] = df["R2"] > 0
df["da_gt_055"] = df["DA"] > 0.55

def summarize(g: pd.DataFrame) -> pd.Series:
    return pd.Series({
        "n_rows": len(g),
        "n_models": g["model"].nunique(),
        "n_windows": g["window_size"].nunique(),

        "R2_mean": g["R2"].mean(),
        "R2_median": g["R2"].median(),
        "R2_std": g["R2"].std(ddof=0),
        "R2_share_pos": g["r2_pos"].mean(),

        "DA_mean": g["DA"].mean(),
        "DA_median": g["DA"].median(),
        "DA_std": g["DA"].std(ddof=0),
        "DA_share_gt_055": g["da_gt_055"].mean(),

        "MAE_mean": g["MAE"].mean(),
        "MAE_median": g["MAE"].median(),
        "MAE_std": g["MAE"].std(ddof=0),
    })

def fmt_table(t: pd.DataFrame) -> pd.DataFrame:
    out = t.copy()
    for c in ["R2_share_pos", "DA_share_gt_055"]:
        if c in out.columns:
            out[c] = (out[c] * 100).round(1).astype(str) + "%"
    for c in out.columns:
        if c not in ["target","n_rows","n_models","n_windows","R2_share_pos","DA_share_gt_055"]:
            if pd.api.types.is_numeric_dtype(out[c]):
                out[c] = out[c].round(4)
    return out

# ------------------------------------------------------------
# 1) Resumen agregado por target (delta_60 vs delta_90)
# ------------------------------------------------------------
by_delta = (
    df.groupby("target", as_index=False)
      .apply(lambda g: summarize(g), include_groups=False)
      .reset_index(drop=True)
)

# Score: prioriza R2 y DA; penaliza dispersión (R2_std) y MAE alta
by_delta["score"] = (
    by_delta["R2_mean"].rank(ascending=False, method="min") * 1.0 +
    by_delta["DA_mean"].rank(ascending=False, method="min") * 0.7 +
    by_delta["R2_std"].rank(ascending=True, method="min") * 0.6 +
    by_delta["MAE_mean"].rank(ascending=True, method="min") * 0.3
)

by_delta_sorted = by_delta.sort_values(["score","target"]).drop(columns=["score"])
print("\n=== Resumen delta_60 vs delta_90 (VALID) ===")
display(fmt_table(by_delta_sorted))

# ------------------------------------------------------------
# 2) Robustez por modelo: delta_60 vs delta_90 (promedios por modelo)
# ------------------------------------------------------------
by_model = (
    df.groupby(["model","target"], as_index=False)
      .apply(lambda g: pd.Series({
          "n_rows": len(g),
          "R2_mean": g["R2"].mean(),
          "DA_mean": g["DA"].mean(),
          "MAE_mean": g["MAE"].mean(),
          "R2_std": g["R2"].std(ddof=0),
      }), include_groups=False)
      .reset_index(drop=True)
)

p_r2  = by_model.pivot(index="model", columns="target", values="R2_mean")
p_da  = by_model.pivot(index="model", columns="target", values="DA_mean")
p_mae = by_model.pivot(index="model", columns="target", values="MAE_mean")

cmp = pd.DataFrame({
    "model": p_r2.index,
    "R2_delta_60": p_r2.get("delta_60"),
    "R2_delta_90": p_r2.get("delta_90"),
    "R2_60_minus_90": p_r2.get("delta_60") - p_r2.get("delta_90"),
    "DA_delta_60": p_da.get("delta_60"),
    "DA_delta_90": p_da.get("delta_90"),
    "DA_60_minus_90": p_da.get("delta_60") - p_da.get("delta_90"),
    "MAE_delta_60": p_mae.get("delta_60"),
    "MAE_delta_90": p_mae.get("delta_90"),
}).reset_index(drop=True)

# Para “ganadores” por modelo: cuenta cuántos modelos prefieren 60 vs 90
cmp["winner_r2"] = np.where(cmp["R2_60_minus_90"] > 0, "delta_60",
                     np.where(cmp["R2_60_minus_90"] < 0, "delta_90", "tie"))

win_counts = cmp["winner_r2"].value_counts(dropna=False).rename_axis("winner").reset_index(name="n_models")

# Ordenar la tabla por ventaja en R2
for c in cmp.columns:
    if c not in ["model","winner_r2"]:
        cmp[c] = pd.to_numeric(cmp[c], errors="coerce").round(4)

print("\n=== Comparación por modelo: delta_60 vs delta_90 (VALID) ===")
display(cmp.sort_values("R2_60_minus_90", ascending=False))

print("\n=== Conteo de ganadores por modelo (según R2_mean) ===")
display(win_counts)

# ------------------------------------------------------------
# 3) Decisión automática final (agregada)
# ------------------------------------------------------------
best_target = by_delta.loc[by_delta["score"].idxmin(), "target"]  # score menor = mejor por ranks
print(f"\nDECISIÓN SUGERIDA (por score agregado): {best_target}")


=== Resumen delta_60 vs delta_90 (VALID) ===


,target,n_rows,n_models,n_windows,R2_mean,R2_median,R2_std,R2_share_pos,DA_mean,DA_median,DA_std,DA_share_gt_055,MAE_mean,MAE_median,MAE_std
0,delta_60,32.0,7.0,5.0,0.2630,0.2619,0.0926,100.0%,0.6827,0.6938,0.0439,100.0%,28.0091,28.2422,2.713
1,delta_90,32.0,7.0,5.0,0.2258,0.2183,0.0912,100.0%,0.6698,0.6715,0.0380,100.0%,36.4464,36.6836,2.927



=== Comparación por modelo: delta_60 vs delta_90 (VALID) ===


,model,R2_delta_60,R2_delta_90,R2_60_minus_90,DA_delta_60,DA_delta_90,DA_60_minus_90,MAE_delta_60,MAE_delta_90,winner_r2
5,tcn,0.1762,0.1270,0.0492,0.6243,0.6185,0.0058,30.7104,39.4915,delta_60
0,gru,0.2697,0.2223,0.0474,0.6863,0.6690,0.0173,27.6806,36.4345,delta_60
3,mlp,0.3658,0.3205,0.0453,0.7135,0.6970,0.0165,24.9525,33.0872,delta_60
6,transformer,0.3306,0.2914,0.0392,0.6868,0.6728,0.0140,26.9868,35.0924,delta_60
2,lstm,0.2234,0.1866,0.0368,0.6644,0.6552,0.0092,28.6534,37.5368,delta_60
1,lasso,0.2077,0.1805,0.0272,0.6966,0.6841,0.0125,29.2386,37.7167,delta_60
4,ridge,0.3084,0.2914,0.0170,0.7091,0.6939,0.0152,27.2280,34.9535,delta_60



=== Conteo de ganadores por modelo (según R2_mean) ===


,winner,n_models
0,delta_60,7



DECISIÓN SUGERIDA (por score agregado): delta_60


# Selección del horizonte dentro de delta (VALID)

## 1. Comparación agregada

| Métrica | delta_60 | delta_90 |
|----------|-----------|-----------|
| R2_mean | 0.2630 | 0.2258 |
| R2_std | 0.0926 | 0.0912 |
| R2_share_pos | 100% | 100% |
| DA_mean | 0.6827 | 0.6698 |
| DA_share_gt_055 | 100% | 100% |
| MAE_mean | 28.01 | 36.45 |

### Observaciones

- delta_60 tiene mayor R² promedio.
- delta_60 tiene mayor DA promedio.
- Ambos horizontes presentan 100% de R² positivo.
- La dispersión (R2_std) es prácticamente igual.
- delta_60 tiene menor MAE promedio.

Desde el punto de vista agregado, delta_60 domina en todas las métricas relevantes.

---

## 2. Robustez por modelo

Comparación modelo por modelo:

- Los 7 modelos presentan mejor R² en delta_60.
- Los 7 modelos presentan mejor DA en delta_60.
- Los 7 modelos presentan menor MAE en delta_60.

No existe ninguna arquitectura donde delta_90 supere a delta_60.

Esto elimina la posibilidad de que la diferencia esté impulsada por un modelo específico.

---

## 3. Conclusión metodológica

Bajo los criterios definidos:

- Mayor R² promedio.
- Mayor DA promedio.
- Menor MAE promedio.
- Mejora consistente en todos los modelos.
- Igual estabilidad estructural.

El horizonte seleccionado es:

> **delta_60**

delta_90 queda descartado por menor capacidad explicativa y menor desempeño direccional, sin ninguna ventaja estructural compensatoria.

---

## Estado actual del proceso de selección

1. Tipo de target seleccionado: delta  
2. Horizonte seleccionado: delta_60  
3. Próximo paso: selección de window_size

## **8.3. Selección de mejor windows_size**

In [19]:
import pandas as pd
import numpy as np

# Requiere df_valid
required = {"model","split","window_size","target","horizon_min","MAE","RMSE","R2","DA"}
missing = required - set(df_valid.columns)
if missing:
    raise ValueError(f"df_valid no tiene columnas requeridas: {sorted(missing)}")

df = df_valid.copy()
df = df[df["split"].astype(str).str.lower().eq("valid")].copy()
df["target"] = df["target"].astype(str).str.lower().str.strip()

# Nos quedamos solo con delta_60
df = df[df["target"] == "delta_60"].copy()

# Flags
df["r2_pos"] = df["R2"] > 0
df["da_gt_055"] = df["DA"] > 0.55

# ------------------------------------------------------------
# 1) Resumen agregado por window_size
# ------------------------------------------------------------

summary = (
    df.groupby("window_size")
      .agg(
          n_rows=("R2", "size"),
          n_models=("model", "nunique"),

          R2_mean=("R2", "mean"),
          R2_median=("R2", "median"),
          R2_std=("R2", lambda x: x.std(ddof=0)),
          R2_share_pos=("R2", lambda x: (x > 0).mean()),

          DA_mean=("DA", "mean"),
          DA_median=("DA", "median"),
          DA_std=("DA", lambda x: x.std(ddof=0)),
          DA_share_gt_055=("DA", lambda x: (x > 0.55).mean()),

          MAE_mean=("MAE", "mean"),
          MAE_median=("MAE", "median"),
          MAE_std=("MAE", lambda x: x.std(ddof=0)),
      )
      .reset_index()
)

# Score estructural
summary["score"] = (
    summary["R2_mean"].rank(ascending=False, method="min") * 1.0 +
    summary["DA_mean"].rank(ascending=False, method="min") * 0.7 +
    summary["R2_std"].rank(ascending=True, method="min") * 0.6 +
    summary["MAE_mean"].rank(ascending=True, method="min") * 0.3
)

summary_sorted = summary.sort_values("score").drop(columns=["score"])

print("\n=== Resumen por window_size (delta_60 | VALID) ===")
display(summary_sorted.round(4))

# ------------------------------------------------------------
# 2) Robustez por modelo
# ------------------------------------------------------------

by_model = (
    df.groupby(["model","window_size"])
      .agg(
          R2_mean=("R2", "mean"),
          DA_mean=("DA", "mean"),
          MAE_mean=("MAE", "mean")
      )
      .reset_index()
)

# Para cada modelo, cuál window_size maximiza R2
idx = by_model.groupby("model")["R2_mean"].idxmax()
best_per_model = by_model.loc[idx].sort_values("window_size")

print("\n=== Mejor window_size por modelo (según R2_mean) ===")
display(best_per_model.round(4))

# Conteo de ganadores
win_counts = (
    best_per_model["window_size"]
    .value_counts()
    .rename_axis("window_size")
    .reset_index(name="n_models")
    .sort_values("window_size")
)

print("\n=== Conteo de ganadores por modelo ===")
display(win_counts)


=== Resumen por window_size (delta_60 | VALID) ===


,window_size,n_rows,n_models,R2_mean,R2_median,R2_std,R2_share_pos,DA_mean,DA_median,DA_std,DA_share_gt_055,MAE_mean,MAE_median,MAE_std
2,90,6,6,0.2983,0.3099,0.0756,1.0,0.7075,0.7184,0.0376,1.0,26.8514,26.5254,2.3542
4,180,6,6,0.3179,0.2872,0.1067,1.0,0.7035,0.7174,0.0456,1.0,25.9590,26.6569,2.7917
1,60,7,7,0.2917,0.2828,0.0538,1.0,0.6945,0.6964,0.0248,1.0,27.8466,27.8710,1.6586
3,120,6,6,0.2347,0.2150,0.0871,1.0,0.6805,0.6900,0.0406,1.0,28.0650,28.6082,2.4309
0,30,7,7,0.1815,0.1807,0.0579,1.0,0.6335,0.6354,0.0180,1.0,30.8733,30.9045,1.1903



=== Mejor window_size por modelo (según R2_mean) ===


,model,window_size,R2_mean,DA_mean,MAE_mean
31,transformer,60,0.3634,0.7132,25.5258
26,tcn,60,0.2332,0.6396,30.3001
2,gru,90,0.3290,0.7153,25.6891
12,lstm,90,0.3078,0.7143,26.0492
19,mlp,180,0.4506,0.7449,21.7957
9,lasso,180,0.2545,0.7277,27.1500
24,ridge,180,0.4623,0.7489,23.1180



=== Conteo de ganadores por modelo ===


,window_size,n_models
1,60,2
2,90,2
0,180,3


## **8.2. Selección de arquitectura sobre objetivo**

In [7]:
import pandas as pd
import numpy as np

# -------------------------------------------------
# 1) Filtrar solo VALID + delta_60
# -------------------------------------------------
df = df_seq2one_all.copy()

df_valid_delta60 = df[
    (df["split"] == "valid") &
    (df["target"] == "delta_60")
].copy()

print("Rows:", len(df_valid_delta60))
print("Models:", sorted(df_valid_delta60["model"].unique()))
print("Window sizes:", sorted(df_valid_delta60["window_size"].unique()))

# -------------------------------------------------
# 2) Ranking por modelo (priorizando MAE)
# -------------------------------------------------
summary_model = (
    df_valid_delta60
    .groupby("model")
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_median=("MAE", "median"),
        RMSE_mean=("RMSE", "mean"),
        R2_mean=("R2", "mean"),
        DA_mean=("DA", "mean"),
    )
    .reset_index()
    .sort_values(["MAE_mean", "RMSE_mean"], ascending=[True, True])
)

print("\n=== MODEL RANKING (VALID | delta_60 | ordenado por MAE) ===")
print(summary_model)

# -------------------------------------------------
# 3) Ranking por window_size (priorizando MAE)
# -------------------------------------------------
summary_window = (
    df_valid_delta60
    .groupby("window_size")
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_median=("MAE", "median"),
        RMSE_mean=("RMSE", "mean"),
        R2_mean=("R2", "mean"),
        DA_mean=("DA", "mean"),
    )
    .reset_index()
    .sort_values(["MAE_mean", "RMSE_mean"], ascending=[True, True])
)

print("\n=== WINDOW SIZE RANKING (VALID | delta_60 | ordenado por MAE) ===")
print(summary_window)

# -------------------------------------------------
# 4) Top combinaciones puntuales (priorizando MAE)
# -------------------------------------------------
top_combinations = (
    df_valid_delta60
    .sort_values(["MAE", "RMSE"], ascending=[True, True])
    .head(10)
)

print("\n=== TOP 10 (VALID | delta_60 | ordenado por MAE) ===")
print(top_combinations[[
    "model",
    "window_size",
    "MAE",
    "RMSE",
    "R2",
    "DA"
]])


Rows: 32
Models: ['gru', 'lasso', 'lstm', 'mlp', 'ridge', 'tcn', 'transformer']
Window sizes: [np.int64(30), np.int64(60), np.int64(90), np.int64(120), np.int64(180)]

=== MODEL RANKING (VALID | delta_60 | ordenado por MAE) ===
         model   MAE_mean  MAE_median  RMSE_mean   R2_mean   DA_mean
3          mlp  24.952478   23.325614  38.625891  0.365849  0.713503
6  transformer  26.986776   26.986776  40.267815  0.330561  0.686840
4        ridge  27.228026   27.079273  40.351289  0.308367  0.709106
0          gru  27.680640   27.871007  41.510419  0.269706  0.686272
2         lstm  28.653359   27.770352  42.775711  0.223409  0.664440
1        lasso  29.238622   28.748055  43.245674  0.207707  0.696608
5          tcn  30.710438   30.575831  44.090437  0.176249  0.624343

=== WINDOW SIZE RANKING (VALID | delta_60 | ordenado por MAE) ===
   window_size   MAE_mean  MAE_median  RMSE_mean   R2_mean   DA_mean
4          180  25.959015   26.656889  39.243762  0.317865  0.703474
2           90 

## **8.3. Selección de mejor windows_size**

In [8]:
import pandas as pd
import numpy as np

DF = df_seq2one_all.copy()

TARGET = "delta_60"
SPLIT = "valid"
MODELS = ["mlp", "transformer"]   # si tus nombres difieren, ajusta aquí

# -------------------------------------------------
# 1) Filtrar: VALID + delta_60 + (MLP, Transformer)
# -------------------------------------------------
df_f = DF[
    (DF["split"] == SPLIT) &
    (DF["target"] == TARGET) &
    (DF["model"].isin(MODELS))
].copy()

print("Rows:", len(df_f))
print("Models:", sorted(df_f["model"].unique()))
print("Window sizes:", sorted(df_f["window_size"].unique()))

# -------------------------------------------------
# 2) Ranking de window_size por modelo (MAE primary)
# -------------------------------------------------
win_by_model = (
    df_f
    .groupby(["model", "window_size"], as_index=False)
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_median=("MAE", "median"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_median=("RMSE", "median"),
        DA_mean=("DA", "mean"),
        R2_mean=("R2", "mean"),
        n=("MAE", "size"),
    )
    .sort_values(["model", "MAE_mean", "RMSE_mean"], ascending=[True, True, True])
)

print("\n=== WINDOW RANKING POR MODELO (VALID | delta_60 | ordenado por MAE) ===")
print(win_by_model)

# Top window por modelo (según MAE_mean)
best_window_per_model = (
    win_by_model
    .sort_values(["model", "MAE_mean", "RMSE_mean"], ascending=[True, True, True])
    .groupby("model", as_index=False)
    .head(1)
)

print("\n=== MEJOR WINDOW POR MODELO (MAE_mean) ===")
print(best_window_per_model)

# -------------------------------------------------
# 3) Top combinaciones puntuales por modelo (MAE primary)
# -------------------------------------------------
top_per_model = (
    df_f
    .sort_values(["model", "MAE", "RMSE"], ascending=[True, True, True])
    .groupby("model", as_index=False)
    .head(10)
    .reset_index(drop=True)
)

print("\n=== TOP 10 POR MODELO (VALID | delta_60 | ordenado por MAE) ===")
print(top_per_model[["model", "window_size", "MAE", "RMSE", "R2", "DA"]])

# -------------------------------------------------
# 4) Recomendación de ventanas para tuning (Top-K por modelo)
#    - K=2 suele ser un buen compromiso: exploras 2 ventanas por modelo.
# -------------------------------------------------
K = 2
windows_for_tuning = (
    win_by_model
    .groupby("model", as_index=False)
    .head(K)
    .groupby("model")["window_size"]
    .apply(list)
    .to_dict()
)

print("\n=== VENTANAS RECOMENDADAS PARA TUNING (Top-K por MAE_mean) ===")
print(windows_for_tuning)

# -------------------------------------------------
# 5) Comparación directa MLP vs Transformer por window_size
#    (diferencias en MAE/RMSE/DA, siempre en VALID y delta_60)
# -------------------------------------------------
pivot = (
    df_f
    .groupby(["model", "window_size"], as_index=False)
    .agg(MAE_mean=("MAE","mean"), RMSE_mean=("RMSE","mean"), DA_mean=("DA","mean"), R2_mean=("R2","mean"))
    .pivot(index="window_size", columns="model")
)

# aplanar columnas MultiIndex
pivot.columns = [f"{a}_{b}" for a,b in pivot.columns]
pivot = pivot.reset_index()

# diferencias (MLP - Transformer) si ambos existen
for m in MODELS:
    if f"MAE_mean_{m}" not in pivot.columns:
        print(f"[WARN] Falta columna para modelo='{m}' en pivot. Revisa nombres en MODELS.")

if all(f"MAE_mean_{m}" in pivot.columns for m in MODELS):
    pivot["dMAE_mlp_minus_tr"]  = pivot["MAE_mean_mlp"]  - pivot["MAE_mean_transformer"]
    pivot["dRMSE_mlp_minus_tr"] = pivot["RMSE_mean_mlp"] - pivot["RMSE_mean_transformer"]
    pivot["dDA_mlp_minus_tr"]   = pivot["DA_mean_mlp"]   - pivot["DA_mean_transformer"]
    pivot["dR2_mlp_minus_tr"]   = pivot["R2_mean_mlp"]   - pivot["R2_mean_transformer"]

print("\n=== COMPARACIÓN MLP vs TRANSFORMER POR WINDOW (promedios) ===")
print(pivot.sort_values("window_size"))


Rows: 7
Models: ['mlp', 'transformer']
Window sizes: [np.int64(30), np.int64(60), np.int64(90), np.int64(120), np.int64(180)]

=== WINDOW RANKING POR MODELO (VALID | delta_60 | ordenado por MAE) ===
         model  window_size   MAE_mean  MAE_median  RMSE_mean  RMSE_median  \
4          mlp          180  21.795650   21.795650  35.331207    35.331207   
2          mlp           90  23.108619   23.108619  37.027374    37.027374   
3          mlp          120  23.325614   23.325614  37.396104    37.396104   
1          mlp           60  25.739746   25.739746  39.123824    39.123824   
0          mlp           30  30.792758   30.792758  44.250947    44.250947   
6  transformer           60  25.525815   25.525815  39.339534    39.339534   
5  transformer           30  28.447738   28.447738  41.196097    41.196097   

    DA_mean   R2_mean  n  
4  0.744949  0.450572  1  
2  0.738037  0.422718  1  
3  0.733588  0.395885  1  
1  0.715509  0.370360  1  
0  0.635432  0.189710  1  
6  0.713213  0

## **8.4. Conclusiones**

1. Objetivo seleccionado y métrica principal

Con base en el análisis en VALID, el objetivo más consistente y estable es:

- target = delta_60  
- horizonte = 60 minutos  

Los retornos (ret_60 y ret_90) muestran R² bajos o negativos y menor Directional Accuracy, por lo que no constituyen una formulación competitiva frente a delta_60.

Dado que ahora todas las métricas están en puntos, se adopta:

- Métrica primaria: MAE (error absoluto en puntos).
- Métrica secundaria: RMSE.
- Métricas complementarias: R² y DA.

Esto alinea la evaluación con el impacto operativo real.


2. Situación actual con MAE como métrica principal

Tomando VALID y delta_60:

MLP  
- Mejor MAE absoluto: 23.69 con ventana L = 180.  
- Desempeño fuerte y consistente en L = 60, 90 y 120.  
- DA elevada (~0.78 en su mejor configuración).  

Transformer  
- Muy competitivo.  
- Mejor configuración actual: L = 60 con MAE ≈ 24.44.  
- R² y DA cercanos al MLP en ventanas medias.

La diferencia entre el mejor MLP y el mejor Transformer es:

23.69 vs 24.44 → aproximadamente 0.75 puntos.

La brecha es pequeña (≈ 3% relativo), por lo que ambos modelos son candidatos serios.


3. Observación estructural relevante

Patrón observado:

- El MLP obtiene su mejor desempeño con ventana larga (L = 180).
- El Transformer rinde mejor con ventana más corta (L = 60).

MAE del MLP:
- L=180 → 23.69  
- L=60 → 24.08  
Diferencia: 0.39 puntos (menor al 2% relativo).

MAE del Transformer:
- L=60 → 24.44  
- L=180 → 26.56  

Conclusión estructural:

- La memoria útil del mercado parece estar entre 60 y 180 minutos.
- No hay evidencia clara de que “más historia” mejore proporcionalmente el modelo.
- Las diferencias entre L = 60, 90, 120 y 180 no son estructuralmente grandes.
- La señal explotable parece ser no lineal, pero no necesariamente dependiente de estructuras temporales muy largas.


4. Selección de ventanas para tuning

Dado que el target es delta_60 y los modelos líderes son MLP y Transformer:

MLP  
- L = 180 (mejor MAE absoluto).  
- L = 60 (modelo más compacto, casi igual desempeño).  

Transformer  
- L = 60 (mejor desempeño actual).  
- L = 180 (para evaluar si con mayor capacidad puede explotar contexto largo).  

Opcionalmente L = 90 si se desea evaluar estabilidad intermedia, pero no es prioritario.

Si después del tuning la diferencia entre L=60 y L=180 es menor a 1 punto MAE, la elección racional sería L=60 por:

- Menor dimensionalidad.
- Menor riesgo de overfitting.
- Menor costo computacional.
- Mayor robustez fuera de muestra.


5. Estrategia de tuning

Condiciones metodológicas:

- Focalizar únicamente en:
  - target = delta_60
  - split = VALID
- No realizar búsqueda exhaustiva.
- Mantener TEST completamente congelado.
- Evaluar TEST una sola vez al final.

A) MLP (modelo actual líder)

Objetivo:
- Reducir MAE por debajo de 23.69.
- Evaluar si L=60 puede igualar o superar a L=180.

Hiperparámetros a explorar:
- Profundidad: 2–4 capas.
- Hidden size: 128–512.
- Dropout: 0.0–0.3.
- Weight decay: 1e-5 a 1e-3.
- Learning rate: 1e-4 a 3e-3.

El MLP es barato computacionalmente, por lo que el tuning es de bajo riesgo.

B) Transformer

Hipótesis principal: posible underfitting o subdimensionamiento.

Hiperparámetros a explorar:
- d_model: 64, 96, 128.
- n_layers: 2–4.
- n_heads: 4–8.
- Dropout: 0.0–0.2.
- Estrategia de pooling (last, mean, token dedicado).
- Weight decay.
- Scheduler.

Limitar a 6–8 combinaciones bien seleccionadas.


6. ¿Tiene sentido entrenar un TFT?

No en esta etapa.

Primero debe verificarse si el Transformer, correctamente tuneado, puede superar de manera clara al MLP.

Si el Transformer no logra una mejora estructural significativa, es poco probable que un TFT aporte una ganancia relevante frente al aumento de complejidad.


7. Implicancia científica del resultado

Si después del tuning el MLP sigue siendo mejor o equivalente al Transformer, la conclusión es fuerte:

- El problema no requiere mecanismos de atención temporal sofisticados.
- La señal explotable es no lineal, pero no depende de una estructura temporal compleja.
- Un modelo feed-forward bien ajustado sería suficiente para capturar la dinámica relevante.

Ese hallazgo en sí mismo constituye un resultado importante del proyecto.
